# Prepare Historical Data

This notebook prepares the historical climate and glacier data for Mer de Glace before training the machine learning model.

The historical data includes:
- Mer de Glace annual mass balance from 1967–2015
- ERA5 monthly temperature from 1967–2015
- ERA5 monthly precipitation from 1967–2015

The final goal is to combine the glacier and climate data into one yearly dataset that can be used to train and test the model.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import xarray as xr

print("Libraries imported successfully.")

## Load Historical Data

The first step is to set the paths to the raw glacier and climate files and make sure each file can be found before working with the data.

In [ ]:
# The notebook is inside data-science/notebooks,
# so move up one level to reach data-science.
DATA_DIR = Path("..") / "data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

mass_balance_file = RAW_DIR / "Bolibar2020_MerDeGlace_MassBalance_1967-2015.csv"
temperature_file = RAW_DIR / "ERA5_MontBlanc_Temperature_1967-2015.nc"
precipitation_file = RAW_DIR / "ERA5_MontBlanc_Precipitation_1967-2015.nc"

print("Mass balance:", mass_balance_file.exists())
print("Temperature:", temperature_file.exists())
print("Precipitation:", precipitation_file.exists())

## Inspect Glacier Mass Balance Data

The Mer de Glace mass balance data will be the value the model is trying to predict. Before using it, the dataset needs to be checked for the correct years, missing values, and data types.

**Source:** Bolibar et al. (2020), reconstructed annual glacier-wide mass balance dataset for glaciers in the French Alps.

In [ ]:
# Load the Mer de Glace mass balance data
mass_balance = pd.read_csv(
    mass_balance_file,
    sep=";",
    header=None,
    names=["year", "mass_balance"]
)

# Display basic information about the dataset
print("Dataset shape:", mass_balance.shape)
print("\nColumns:")
print(mass_balance.columns.tolist())

mass_balance.head()

In [ ]:
# Check the mass balance dataset for missing values and year coverage
print("Year range:", mass_balance["year"].min(), "-", mass_balance["year"].max())

print("\nMissing values:")
print(mass_balance.isnull().sum())

print("\nData types:")
print(mass_balance.dtypes)

## Inspect Historical Temperature Data

The ERA5 temperature file contains monthly 2-meter temperature data for the Mont Blanc region from 1967–2015. The file structure, dates, coordinates, and units need to be checked before processing the values.

**Source:** ERA5 historical climate data from the Copernicus Climate Data Store.

In [ ]:
# Load the ERA5 historical temperature dataset
temperature = xr.open_dataset(temperature_file)

# Display the dataset structure
temperature

# Check the temperature variable information
print("Temperature units:", temperature["t2m"].attrs.get("units"))
print("Temperature long name:", temperature["t2m"].attrs.get("long_name"))

print("\nMinimum temperature:", float(temperature["t2m"].min()))
print("Maximum temperature:", float(temperature["t2m"].max()))

## Inspect Historical Precipitation Data

The ERA5 precipitation file contains monthly total precipitation data for the Mont Blanc region from 1967–2015. The file structure, dates, coordinates, and units need to be checked before processing the values.

**Source:** ERA5 historical climate data from the Copernicus Climate Data Store.

In [ ]:
# Load the ERA5 historical precipitation dataset
precipitation = xr.open_dataset(precipitation_file)

# Display the dataset structure
precipitation

# Check the precipitation variable information
print("Precipitation units:", precipitation["tp"].attrs.get("units"))
print("Precipitation long name:", precipitation["tp"].attrs.get("long_name"))

print("\nMinimum precipitation:", float(precipitation["tp"].min()))
print("Maximum precipitation:", float(precipitation["tp"].max()))

# Display the ERA5 grid coordinates
print("Latitudes:")
print(temperature["latitude"].values)

print("\nLongitudes:")
print(temperature["longitude"].values)

print("\nFirst month temperature values (K):")
print(temperature["t2m"].isel(valid_time=0).values)

## Process Historical Temperature Data

The ERA5 download contains four grid points around the Mont Blanc region. The four points are averaged to create one regional temperature value for each month instead of using a single grid point. The temperature values are also converted from Kelvin to Celsius.

This gives the model a consistent regional climate measurement for the area around Mer de Glace.

**Source:** ERA5 historical climate data from the Copernicus Climate Data Store.

In [ ]:
# Average the four ERA5 grid points
monthly_temperature = temperature["t2m"].mean(
    dim=["latitude", "longitude"]
)

# Convert temperature from Kelvin to Celsius
monthly_temperature_c = monthly_temperature - 273.15

# Display the first 12 monthly values
monthly_temperature_c.isel(valid_time=slice(0, 12))

## Create Climate Features

The monthly climate data needs to be reduced to a few useful features before training the model.

Warm-season temperature will use the average temperature from May through August. This represents the main melt period and is supported by previous Alpine glacier mass balance research.

Cold-season precipitation will use the total precipitation from October through February. This represents winter accumulation.

Annual mean temperature will also be calculated for comparison, but it will only be included in the final model if testing shows that it improves the predictions.

**Source:** Van der Meer et al. (2025), The Cryosphere — a machine learning glacier mass balance study that used May–August temperature and October–February precipitation as climate predictors.

In [ ]:
# Add year and month coordinates from the time values
monthly_temperature_c = monthly_temperature_c.assign_coords(
    year=monthly_temperature_c["valid_time"].dt.year,
    month=monthly_temperature_c["valid_time"].dt.month
)

# Calculate annual mean temperature for comparison
annual_mean_temperature = monthly_temperature_c.groupby("year").mean()

# Calculate warm-season mean temperature (May-August)
warm_season_temperature = monthly_temperature_c.where(
    monthly_temperature_c["month"].isin([5, 6, 7, 8]),
    drop=True
).groupby("year").mean()

print("Annual mean temperature:")
print(annual_mean_temperature.isel(year=slice(0, 5)).values)

print("\nWarm-season mean temperature:")
print(warm_season_temperature.isel(year=slice(0, 5)).values)

## Process Historical Precipitation Data

The four ERA5 grid points are averaged to create one regional precipitation value for each month.

ERA5 monthly averaged total precipitation represents an average daily accumulation. To get the total precipitation for each month, the values are converted from meters to millimeters and multiplied by the number of days in the month.

Cold-season precipitation will later use October through February to represent winter accumulation.

**Source:** ERA5 documentation from ECMWF/Copernicus. Monthly averaged total precipitation must be multiplied by the number of days in the month to calculate monthly precipitation totals.

In [ ]:
# Average the four ERA5 grid points
monthly_precipitation = precipitation["tp"].mean(
    dim=["latitude", "longitude"]
)

# Get the number of days in each month
days_in_month = monthly_precipitation["valid_time"].dt.days_in_month

# Convert ERA5 monthly averaged precipitation from m/day
# to total precipitation for the month in mm
monthly_precipitation_mm = (
    monthly_precipitation * 1000 * days_in_month
)

# Display the first 12 monthly totals
monthly_precipitation_mm.isel(valid_time=slice(0, 12))

# Display the first 12 monthly values
monthly_precipitation_mm.isel(valid_time=slice(0, 12))

## Create Cold-Season Precipitation Feature

Cold-season precipitation uses October through February. Since these months cross two calendar years, October through December are assigned to the following year.

For example, the 1968 cold season includes October–December 1967 and January–February 1968. This keeps the winter accumulation period matched with the following warm season and annual mass balance year.

**Source:** Van der Meer et al. (2025), *The Cryosphere* — October–February precipitation was used as a winter accumulation predictor in an Alpine glacier mass balance model.

In [ ]:
# Get the year and month for each precipitation value
precip_year = monthly_precipitation_mm["valid_time"].dt.year
precip_month = monthly_precipitation_mm["valid_time"].dt.month

# Assign October-December to the following year's cold season
cold_season_year = xr.where(
    precip_month >= 10,
    precip_year + 1,
    precip_year
)

# Keep only October-February
cold_season_precipitation = monthly_precipitation_mm.where(
    precip_month.isin([10, 11, 12, 1, 2]),
    drop=True
)

# Add the matching cold-season year
cold_season_precipitation = cold_season_precipitation.assign_coords(
    cold_season_year=cold_season_year.where(
        precip_month.isin([10, 11, 12, 1, 2]),
        drop=True
    )
)

# Total precipitation for each cold season
cold_season_precipitation = cold_season_precipitation.groupby(
    "cold_season_year"
).sum()

cold_season_precipitation

# Keep only complete cold seasons from 1968-2015
cold_season_precipitation = cold_season_precipitation.sel(
    cold_season_year=slice(1968, 2015)
)

print("Cold-season year range:",
      int(cold_season_precipitation["cold_season_year"].min()),
      "-",
      int(cold_season_precipitation["cold_season_year"].max()))

print("Number of complete cold seasons:",
      cold_season_precipitation.sizes["cold_season_year"])

print("\nFirst five cold-season totals:")
print(cold_season_precipitation.isel(cold_season_year=slice(0, 5)).values)

## Build Historical Climate Table

The processed temperature and precipitation features are converted into one yearly table so they can be matched with the glacier mass balance data by year.

In [ ]:
# Convert annual temperature features to pandas Series
annual_temp_series = annual_mean_temperature.to_series()
warm_temp_series = warm_season_temperature.to_series()

# Convert cold-season precipitation to pandas Series
cold_precip_series = cold_season_precipitation.to_series()

# Build one climate DataFrame
climate_data = pd.DataFrame({
    "annual_mean_temp_c": annual_temp_series,
    "warm_season_temp_c": warm_temp_series
})

# Add cold-season precipitation using the matching year
climate_data["cold_season_precip_mm"] = cold_precip_series

# Keep only years with complete climate data
climate_data = climate_data.dropna()

climate_data.head()

# Check the completed climate dataset
print("Dataset shape:", climate_data.shape)
print("Year range:", climate_data.index.min(), "-", climate_data.index.max())

print("\nMissing values:")
print(climate_data.isnull().sum())

print("\nData types:")
print(climate_data.dtypes)

## Combine Climate and Glacier Data

The processed climate features are matched with the Mer de Glace annual mass balance data by year. This creates the final historical dataset that can be used for model training and testing.

In [ ]:
# Make year a column in the climate table
climate_data_reset = climate_data.reset_index()

# Merge climate features with glacier mass balance
historical_data = pd.merge(
    climate_data_reset,
    mass_balance,
    on="year",
    how="inner"
)

historical_data.head()

## Check Final Historical Dataset

The combined dataset needs one final check before saving it. This makes sure the climate features and mass balance data cover the same years and that no missing values were introduced during the merge.

In [ ]:
# Check the final combined historical dataset
print("Dataset shape:", historical_data.shape)
print("Year range:", historical_data["year"].min(), "-", historical_data["year"].max())

print("\nMissing values:")
print(historical_data.isnull().sum())

print("\nData types:")
print(historical_data.dtypes)

## Save Processed Historical Data

The finished historical dataset is saved as a CSV so it can be used later for model training without repeating the raw data processing steps.

In [32]:
# Set the output file path
historical_output_file = (
    PROCESSED_DIR / "MerDeGlace_Historical_Training_Data_1968-2015.csv"
)

# Save the processed dataset
historical_data.to_csv(
    historical_output_file,
    index=False
)

print("Saved to:", historical_output_file)
print("File exists:", historical_output_file.exists())

Saved to: ../data/processed/MerDeGlace_Historical_Training_Data_1968-2015.csv
File exists: True
